# LeetCode 2240 — Number of Ways to Buy Pens and Pencils bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-buy-pens-and-pencils/ · Độ khó: **Medium** · Chủ đề: Math, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def waysToBuyPensPencils(self, total: int, cost1: int, cost2: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa: đếm số cặp nguyên không âm `(x, y)` với `cost1·x + cost2·y ≤ total` — **đếm điểm nguyên dưới một đường thẳng**.
2. Giảm từ hai vòng lặp (`O(total²/(c1·c2))`) xuống **một vòng lặp** + phép chia nguyên (`O(total/c1)`).
3. Mẹo **duyệt theo món đắt hơn** để giảm số vòng lặp.
4. Biết thuật toán **floor_sum** `O(log)` (kiểu Euclid) tính `Σ ⌊(a·i + b)/m⌋` — đếm điểm nguyên siêu nhanh.
5. Kiểm thử bằng brute force và tránh lỗi "quên trường hợp mua 0 món".

## 2. Đề bài & ví dụ

Có `total` tiền; một bút mực giá `cost1`, một bút chì giá `cost2`. Có thể tiêu một phần hoặc toàn bộ tiền để mua **số lượng bất kỳ** (kể cả 0) mỗi loại. Trả về số **cách khác nhau** để mua (mỗi cách là một cặp (số bút mực, số bút chì)).

**Ví dụ 1:** `total = 20, cost1 = 10, cost2 = 5` → `9`. Mua 0 bút mực: 0..4 bút chì (5 cách); 1 bút mực: 0..2 (3 cách); 2 bút mực: 0 (1 cách) ⇒ `5 + 3 + 1 = 9`.

**Ví dụ 2:** `total = 5, cost1 = 10, cost2 = 10` → `1` (chỉ có cách mua 0 + 0).

**Ràng buộc:** `1 ≤ total, cost1, cost2 ≤ 10^6`.

In [1]:
import random
import time
from typing import List

examples = [(20, 10, 5, 9), (5, 10, 10, 1)]
for total, c1, c2, out in examples:
    print(f"total={total}, cost1={c1}, cost2={c2} -> {out}")

total=20, cost1=10, cost2=5 -> 9
total=5, cost1=10, cost2=10 -> 1


## 3. Kiến thức nền

### 3.1 Chia nguyên `//`

Với tiền còn lại `rem ≥ 0`, số bút chì tối đa là `rem // cost2`; số lựa chọn số bút chì là `rem // cost2 + 1` (tính cả 0).

In [2]:
for rem in [0, 4, 5, 9, 10, 20]:
    print(f"còn {rem:2d} tiền, giá 5: mua tối đa {rem // 5} bút chì -> {rem // 5 + 1} lựa chọn (0..{rem // 5})")

còn  0 tiền, giá 5: mua tối đa 0 bút chì -> 1 lựa chọn (0..0)
còn  4 tiền, giá 5: mua tối đa 0 bút chì -> 1 lựa chọn (0..0)
còn  5 tiền, giá 5: mua tối đa 1 bút chì -> 2 lựa chọn (0..1)
còn  9 tiền, giá 5: mua tối đa 1 bút chì -> 2 lựa chọn (0..1)
còn 10 tiền, giá 5: mua tối đa 2 bút chì -> 3 lựa chọn (0..2)
còn 20 tiền, giá 5: mua tối đa 4 bút chì -> 5 lựa chọn (0..4)


### 3.2 Đếm điểm nguyên trong tam giác

Tập `{(x, y) ∈ ℤ≥0² : c1·x + c2·y ≤ T}` là các điểm nguyên trong tam giác vuông với hai cạnh góc vuông dài `T/c1` và `T/c2`. Diện tích xấp xỉ `T²/(2·c1·c2)` ⇒ đáp án có thể tới `~5·10^11` khi `c1 = c2 = 1, T = 10^6` — Python `int` không tràn, nhưng các ngôn ngữ khác cần `long long`.

In [3]:
T = 10**6
approx = T * T // 2
exact = sum(T - x + 1 for x in range(T + 1))     # c1 = c2 = 1
print("xấp xỉ diện tích:", approx, "| chính xác:", exact, "| vượt int32?", exact > 2**31 - 1)

xấp xỉ diện tích: 500000000000 | chính xác: 500001500001 | vượt int32? True


### 3.3 Công thức tổng `0 + 1 + … + N`

Khi `cost1 = cost2 = 1`: đáp án là `Σ_{x=0}^{T} (T − x + 1) = (T+1)(T+2)/2`. Đây là trường hợp kiểm tra nhanh cho mọi lời giải.

In [4]:
for T in [0, 1, 2, 5, 10]:
    print(T, "->", (T + 1) * (T + 2) // 2)

0 -> 1
1 -> 3
2 -> 6
5 -> 21
10 -> 66


## 4. Ý tưởng nền tảng / trực giác

Lập bảng theo số bút mực `x` (ví dụ 1):

| x (bút mực) | tiền còn | bút chì tối đa | số cách |
|---|---|---|---|
| 0 | 20 | 4 | 5 |
| 1 | 10 | 2 | 3 |
| 2 | 0 | 0 | 1 |
| | | **tổng** | **9** |

Với mỗi `x` cố định, **không cần** liệt kê từng `y` — số cách là `(total − c1·x) // c2 + 1`. Chỉ còn một vòng lặp theo `x` từ 0 tới `total // c1`.

Hình học: mỗi `x` là một **cột** điểm nguyên dưới đường thẳng `c1·x + c2·y = total`; chiều cao cột là `⌊(total − c1·x)/c2⌋ + 1`.

In [5]:
total, c1, c2 = 20, 10, 5
for x in range(total // c1 + 1):
    h = (total - c1 * x) // c2 + 1
    print(f"x={x}: " + "● " * h + f"({h} điểm: y = 0..{h - 1})")

x=0: ● ● ● ● ● (5 điểm: y = 0..4)
x=1: ● ● ● (3 điểm: y = 0..2)
x=2: ● (1 điểm: y = 0..0)


### 4.1 Nhìn theo hàng thay vì theo cột

Cùng tập điểm có thể đếm theo **hàng** (cố định `y` = số bút chì): mỗi hàng có `⌊(T − c2·y)/c1⌋ + 1` điểm. Hai cách đếm cho cùng kết quả (đếm một tập theo hai chiều) — đây chính là cơ sở của mẹo "duyệt theo món đắt hơn" (ít cột/hàng hơn) và của bước "đổi vai trục" trong floor_sum.

In [6]:
total, c1, c2 = 20, 10, 5
by_col = [(total - c1 * x) // c2 + 1 for x in range(total // c1 + 1)]
by_row = [(total - c2 * y) // c1 + 1 for y in range(total // c2 + 1)]
print("đếm theo cột (x):", by_col, "=", sum(by_col), "| số vòng:", len(by_col))
print("đếm theo hàng (y):", by_row, "=", sum(by_row), "| số vòng:", len(by_row))
for y in range(total // c2, -1, -1):
    print(f"y={y}: " + "".join("●" if c1 * x + c2 * y <= total else "·" for x in range(total // c1 + 1)))

đếm theo cột (x): [5, 3, 1] = 9 | số vòng: 3
đếm theo hàng (y): [3, 2, 2, 1, 1] = 9 | số vòng: 5
y=4: ●··
y=3: ●··
y=2: ●●·
y=1: ●●·
y=0: ●●●


### 4.2 Ước lượng độ lớn đáp án

Số điểm nguyên trong tam giác xấp xỉ diện tích `T²/(2·c1·c2)` cộng phần biên `~ T/(2c1) + T/(2c2)`. Điều này giúp kiểm tra nhanh: nếu kết quả lệch xa xấp xỉ thì nhiều khả năng có bug (thiếu `+1`, thiếu cột cuối…).

In [7]:
for T, a, b in [(20, 10, 5), (1000, 7, 3), (10**6, 13, 17)]:
    approx = T * T / (2 * a * b) + T / (2 * a) + T / (2 * b) + 1
    print(f"T={T}, c1={a}, c2={b}: chính xác = {ways_one_loop(T, a, b) if 'ways_one_loop' in globals() else sum((T - a * x) // b + 1 for x in range(T // a + 1))}, xấp xỉ ≈ {approx:.0f}")

T=20, c1=10, c2=5: chính xác = 9, xấp xỉ ≈ 8
T=1000, c1=7, c2=3: chính xác = 24072, xấp xỉ ≈ 24049
T=1000000, c1=13, c2=17: chính xác = 2262513575, xấp xỉ ≈ 2262511313


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp

**Ý tưởng:** duyệt mọi `x`, mọi `y`, đếm cặp có chi phí `≤ total`.

In [8]:
def ways_brute(total, cost1, cost2):
    cnt = 0
    for x in range(total // cost1 + 1):
        for y in range(total // cost2 + 1):
            if cost1 * x + cost2 * y <= total:
                cnt += 1
    return cnt

for total, c1, c2, out in examples:
    got = ways_brute(total, c1, c2); print(total, c1, c2, "->", got); assert got == out

20 10 5 -> 9
5 10 10 -> 1


**Độ phức tạp:** `O((T/c1)·(T/c2))` — với `c1 = c2 = 1, T = 10^6` là `10^12` phép thử. Space `O(1)`.

**Nút thắt:** vòng lặp trong chỉ để **đếm** các `y` liên tiếp `0, 1, …, ⌊rem/c2⌋` — đếm được bằng một phép chia.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** với `x` cố định, các `y` hợp lệ tạo thành dãy liên tiếp `0..⌊(T − c1·x)/c2⌋` ⇒ số lượng `= ⌊(T − c1·x)/c2⌋ + 1`. Đáp án:

$$\text{ans} = \sum_{x=0}^{\lfloor T/c_1 \rfloor} \left( \left\lfloor \frac{T - c_1 x}{c_2} \right\rfloor + 1 \right).$$

**Insight 2 (đối xứng):** đổi vai `c1 ↔ c2` không đổi đáp án ⇒ duyệt theo món **đắt hơn** để số vòng lặp là `T / max(c1, c2)`.

**Insight 3 (floor_sum):** đặt `N = ⌊T/c1⌋`, `r = T mod c1`, đổi biến `i = N − x`: `T − c1·x = r + c1·i`. Tổng thành `Σ_{i=0}^{N} ⌊(c1·i + r)/c2⌋ + (N + 1)` — đúng dạng `floor_sum(n, m, a, b) = Σ_{i=0}^{n−1} ⌊(a·i + b)/m⌋`, tính được trong `O(log)` bằng thuật toán kiểu Euclid.

**Quy tắc "nếu… thì…":**
- **Nếu** vòng trong chỉ đếm một dãy liên tiếp **thì** thay bằng công thức.
- **Nếu** bài đối xứng giữa hai tham số **thì** chọn tham số cho ít vòng lặp hơn.
- **Nếu** cần `Σ ⌊(a·i + b)/m⌋` với `n` cực lớn **thì** floor_sum.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Một vòng lặp + chia nguyên (bản nộp chính)

In [9]:
def ways_one_loop(total, cost1, cost2):
    ans = 0
    for x in range(total // cost1 + 1):
        ans += (total - cost1 * x) // cost2 + 1
    return ans

for total, c1, c2, out in examples:
    assert ways_one_loop(total, c1, c2) == out
print([ways_one_loop(t, a, b) for t, a, b, _ in examples])

[9, 1]


**Độ phức tạp Cách 2:** `O(T / c1)` ≤ `10^6` vòng — chạy tốt trong Python (~0.1–0.3 s trường hợp xấu nhất). Space `O(1)`.

### 7.2 Cách 3 — Duyệt theo món đắt hơn

In [10]:
def ways_pricier(total, cost1, cost2):
    if cost1 < cost2:
        cost1, cost2 = cost2, cost1          # duyệt theo món đắt hơn -> ít vòng hơn
    return sum((total - cost1 * x) // cost2 + 1 for x in range(total // cost1 + 1))

for total, c1, c2, out in examples:
    assert ways_pricier(total, c1, c2) == out
print([ways_pricier(t, a, b) for t, a, b, _ in examples])

[9, 1]


**Độ phức tạp Cách 3:** `O(T / max(c1, c2))`. Trường hợp xấu vẫn là `c1 = c2 = 1` (`10^6` vòng), nhưng trung bình nhanh hơn nhiều.

### 7.3 Cách 4 — floor_sum `O(log)`

Thuật toán (như trong thư viện AtCoder): với `0 ≤ a, b` có thể rút phần nguyên `a // m`, `b // m` ra ngoài (đóng góp cấp số cộng), rồi đổi vai trục `x ↔ y` — giống thuật toán Euclid nên số bước là `O(log m)`.

In [11]:
def floor_sum(n, m, a, b):
    """Σ_{i=0}^{n-1} floor((a*i + b) / m), với n >= 0, m >= 1, a, b >= 0."""
    ans = 0
    while True:
        if a >= m:
            ans += (n - 1) * n // 2 * (a // m); a %= m
        if b >= m:
            ans += n * (b // m); b %= m
        y_max = a * n + b
        if y_max < m:
            return ans
        n, b, m, a = y_max // m, y_max % m, a, m      # đổi vai hai trục

def ways_floor_sum(total, cost1, cost2):
    N, r = divmod(total, cost1)
    return floor_sum(N + 1, cost2, cost1, r) + (N + 1)

def floor_sum_naive(n, m, a, b):
    return sum((a * i + b) // m for i in range(n))

rng = random.Random(0)
for _ in range(2000):
    n, m, a, b = rng.randint(0, 30), rng.randint(1, 20), rng.randint(0, 40), rng.randint(0, 40)
    assert floor_sum(n, m, a, b) == floor_sum_naive(n, m, a, b)
for total, c1, c2, out in examples:
    assert ways_floor_sum(total, c1, c2) == out
print("floor_sum khớp bản ngây thơ;", [ways_floor_sum(t, a, b) for t, a, b, _ in examples])

floor_sum khớp bản ngây thơ; [9, 1]


**Độ phức tạp Cách 4:** `O(log max(c1, c2))` — không phụ thuộc `T`; dùng được cả khi `T ~ 10^18`. Space `O(1)`.

### 7.4 Cách 5 — Gom theo nhóm thương (giữa Cách 2 và Cách 4)

Khi `x` tăng 1, tiền còn giảm `c1`, nên `⌊rem / c2⌋` giảm theo các bước đều đặn nếu `c1` là bội của `c2`. Trường hợp tổng quát, ta có thể nhảy nhiều `x` cùng lúc khi biết `c2 | c1`: số cách là cấp số cộng. Đây là bản đặc biệt — hữu ích để hiểu vì sao floor_sum "rút phần nguyên `a // m`".

In [12]:
def ways_divisible(total, cost1, cost2):
    """Chỉ khi cost1 % cost2 == 0: chiều cao cột giảm đều q = cost1 // cost2 mỗi bước."""
    assert cost1 % cost2 == 0
    N = total // cost1; q = cost1 // cost2
    first = total // cost2 + 1                    # x = 0
    return (N + 1) * first - q * N * (N + 1) // 2 # cấp số cộng first, first - q, ...

for t, a, b in [(20, 10, 5), (100, 6, 3), (7, 4, 2), (10**6, 2, 1)]:
    assert ways_divisible(t, a, b) == ways_one_loop(t, a, b)
print("cấp số cộng khớp khi c2 | c1:", ways_divisible(20, 10, 5), ways_divisible(10**6, 2, 1))

cấp số cộng khớp khi c2 | c1: 9 250001000001


## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 (`total = 20, cost1 = 10, cost2 = 5`):

| x | `total − 10x` | `// 5` | `+1` | ans |
|---|---|---|---|---|
| 0 | 20 | 4 | 5 | 5 |
| 1 | 10 | 2 | 3 | 8 |
| 2 | 0 | 0 | 1 | 9 |

In [13]:
def trace_one_loop(total, cost1, cost2):
    ans = 0
    for x in range(total // cost1 + 1):
        rem = total - cost1 * x; add = rem // cost2 + 1; ans += add
        print(f"x={x}: còn {rem}, bút chì 0..{rem // cost2} -> +{add}, ans = {ans}")
    return ans

assert trace_one_loop(20, 10, 5) == 9

x=0: còn 20, bút chì 0..4 -> +5, ans = 5
x=1: còn 10, bút chì 0..2 -> +3, ans = 8
x=2: còn 0, bút chì 0..0 -> +1, ans = 9


In [14]:
assert trace_one_loop(5, 10, 10) == 1

x=0: còn 5, bút chì 0..0 -> +1, ans = 1


Trace Cách 4 (floor_sum) trên ví dụ 1: `N = 2`, `r = 0`, cần `floor_sum(3, 5, 10, 0) + 3`. Ta in từng vòng của thuật toán:

In [15]:
def trace_floor_sum(n, m, a, b):
    ans = 0; step = 0
    while True:
        step += 1
        print(f"vòng {step}: n={n}, m={m}, a={a}, b={b}, ans={ans}")
        if a >= m:
            ans += (n - 1) * n // 2 * (a // m); print(f"  rút a//m = {a // m}: +{(n - 1) * n // 2 * (a // m)}"); a %= m
        if b >= m:
            ans += n * (b // m); print(f"  rút b//m = {b // m}: +{n * (b // m)}"); b %= m
        y_max = a * n + b
        if y_max < m:
            print("  dừng, kết quả =", ans); return ans
        n, b, m, a = y_max // m, y_max % m, a, m

v = trace_floor_sum(3, 5, 10, 0)
print("ans =", v, "+ (N+1) =", v + 3)

vòng 1: n=3, m=5, a=10, b=0, ans=0
  rút a//m = 2: +6
  dừng, kết quả = 6
ans = 6 + (N+1) = 9


In [16]:
# Một ví dụ tự dựng không chia hết (đã kiểm bằng brute force):
v = trace_floor_sum(*(lambda N, r: (N + 1, 7, 13, r))(*divmod(100, 13)))
print("ways(100, 13, 7) =", v + 100 // 13 + 1, "| brute:", ways_brute(100, 13, 7))
assert v + 100 // 13 + 1 == ways_brute(100, 13, 7)

vòng 1: n=8, m=7, a=13, b=9, ans=0
  rút a//m = 1: +28
  rút b//m = 1: +8
vòng 2: n=7, m=6, a=7, b=1, ans=36
  rút a//m = 1: +21
vòng 3: n=1, m=1, a=6, b=2, ans=57
  rút a//m = 6: +0
  rút b//m = 2: +2
  dừng, kết quả = 59
ways(100, 13, 7) = 67 | brute: 67


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def waysToBuyPensPencils(self, total: int, cost1: int, cost2: int) -> int:
        if cost1 < cost2:
            cost1, cost2 = cost2, cost1              # duyệt theo món đắt hơn
        ans = 0
        for x in range(total // cost1 + 1):          # x = số món đắt, 0..total//cost1
            ans += (total - cost1 * x) // cost2 + 1  # số món rẻ: 0..rem//cost2
        return ans

for total, c1, c2, out in examples:
    assert Solution().waysToBuyPensPencils(total, c1, c2) == out
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Song ánh.** Mỗi cách mua ứng với đúng một cặp `(x, y) ∈ ℤ≥0²` với `c1·x + c2·y ≤ T` (x bút mực, y bút chì) và ngược lại. Phân hoạch tập này theo `x`:
- `x` phải thỏa `c1·x ≤ T` (vì `y ≥ 0`) ⇒ `0 ≤ x ≤ ⌊T/c1⌋` — đúng miền của vòng lặp.
- Với `x` cố định, `y` hợp lệ ⇔ `0 ≤ y ≤ (T − c1·x)/c2` ⇔ `0 ≤ y ≤ ⌊(T − c1·x)/c2⌋` (vì `y` nguyên) — có đúng `⌊(T − c1·x)/c2⌋ + 1` giá trị.

Cộng theo các lớp rời nhau ⇒ đúng tổng số cặp.

**Đổi vai (Cách 3).** Ánh xạ `(x, y) ↦ (y, x)` là song ánh giữa tập ứng với `(c1, c2)` và tập ứng với `(c2, c1)` ⇒ đáp án đối xứng.

**floor_sum (Cách 4).** Với `i = N − x`, `T − c1·x = T − c1·N + c1·i = r + c1·i`, `i` chạy `0..N`. Bước "rút phần nguyên": `⌊((a mod m + m·(a//m))·i + b)/m⌋ = (a//m)·i + ⌊((a mod m)·i + b)/m⌋`, cộng theo `i` ra cấp số cộng. Khi `a, b < m`, tổng đếm điểm nguyên dưới đoạn thẳng; đếm theo **hàng** thay vì cột cho bài toán cùng dạng với `(a, m)` đổi chỗ — như Euclid, `m` giảm nhanh nên vòng lặp kết thúc sau `O(log)` bước. Bộ test kiểm floor_sum với bản ngây thơ.

### 10.1 Kiểm chứng đổi biến của floor_sum bằng số

Ta in hai dãy: chiều cao cột theo `x` (từ trái sang phải) và `⌊(c1·i + r)/c2⌋ + 1` theo `i = N − x` — phải là cùng một dãy đảo ngược.

In [18]:
for T, a, b in [(20, 10, 5), (100, 13, 7), (37, 4, 9)]:
    N, r = divmod(T, a)
    cols = [(T - a * x) // b + 1 for x in range(N + 1)]
    via_i = [(a * i + r) // b + 1 for i in range(N + 1)]
    print(f"T={T}, c1={a}, c2={b}: cột theo x = {cols} | theo i = {via_i} | đảo ngược khớp? {cols[::-1] == via_i}")
    assert cols[::-1] == via_i

T=20, c1=10, c2=5: cột theo x = [5, 3, 1] | theo i = [1, 3, 5] | đảo ngược khớp? True
T=100, c1=13, c2=7: cột theo x = [15, 13, 11, 9, 7, 6, 4, 2] | theo i = [2, 4, 6, 7, 9, 11, 13, 15] | đảo ngược khớp? True
T=37, c1=4, c2=9: cột theo x = [5, 4, 4, 3, 3, 2, 2, 2, 1, 1] | theo i = [1, 1, 2, 2, 2, 3, 3, 4, 4, 5] | đảo ngược khớp? True


### 10.2 Vì sao floor_sum kết thúc nhanh?

Sau bước rút gọn, `a < m` và `b < m`. Bước đổi vai đặt `m_mới = a_cũ < m_cũ` và `a_mới = m_cũ`; vòng sau rút `a_mới // m_mới` và giữ `a_mới mod m_mới` — đúng một bước của thuật toán Euclid trên cặp `(m, a)`. Euclid trên số `≤ 10^6` cần không quá ~30 bước, nên floor_sum gần như `O(1)` trong thực tế.

In [19]:
def floor_sum_steps(n, m, a, b):
    steps = 0
    while True:
        steps += 1
        if a >= m: a %= m
        if b >= m: b %= m
        y_max = a * n + b
        if y_max < m: return steps
        n, b, m, a = y_max // m, y_max % m, a, m

worst = max(floor_sum_steps(10**6 // c1 + 1, c2, c1, 10**6 % c1) for c1, c2 in [(832040, 514229), (10**6, 999999), (3, 7), (1, 1), (999983, 2)])
print("số vòng lặp floor_sum lớn nhất trên vài cặp khó (gồm cặp Fibonacci):", worst)

số vòng lặp floor_sum lớn nhất trên vài cặp khó (gồm cặp Fibonacci): 7


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng | `O((T/c1)·(T/c2))` | `O(1)` |
| 2. Một vòng | `O(T/c1)` | `O(1)` |
| 3. Món đắt hơn | `O(T/max(c1, c2))` | `O(1)` |
| 4. floor_sum | `O(log max(c1, c2))` | `O(1)` |
| 5. Cấp số cộng (khi `c2 | c1`) | `O(1)` | `O(1)` |

Đo trường hợp xấu nhất `T = 10^6, c1 = c2 = 1`:

In [20]:
for name, fn in [("one_loop", ways_one_loop), ("pricier", ways_pricier), ("floor_sum", ways_floor_sum)]:
    t = time.perf_counter(); v = fn(10**6, 1, 1)
    print(f"{name:9s}: {v} trong {time.perf_counter() - t:.4f}s")
t = time.perf_counter(); v = ways_floor_sum(10**18, 3, 7)
print(f"floor_sum với T = 10^18: {v} trong {time.perf_counter() - t:.6f}s")
assert ways_one_loop(10**6, 1, 1) == (10**6 + 1) * (10**6 + 2) // 2

one_loop : 500001500001 trong 0.0914s


pricier  : 500001500001 trong 0.0620s
floor_sum: 500001500001 trong 0.0000s
floor_sum với T = 10^18: 23809523809523809785714285714285715 trong 0.000076s


## 12. Bộ test

So mọi cách với brute force (dữ liệu nhỏ), với công thức đóng `(T+1)(T+2)/2` khi giá 1, và giữa các cách nhanh với nhau (dữ liệu lớn). Seed cố định.

In [21]:
fns = {"one_loop": ways_one_loop, "pricier": ways_pricier, "floor_sum": ways_floor_sum,
       "Solution": lambda t, a, b: Solution().waysToBuyPensPencils(t, a, b)}
edge = [(1, 1, 1, 3), (1, 2, 2, 1), (1, 1, 2, 2), (2, 1, 1, 6), (10, 3, 3, 10), (10**6, 10**6, 10**6, 3), (999999, 10**6, 1, 10**6)]
for total, c1, c2, out in examples + edge:
    if total <= 1000: assert ways_brute(total, c1, c2) == out, (total, c1, c2)
    for name, fn in fns.items():
        assert fn(total, c1, c2) == out, (name, total, c1, c2, fn(total, c1, c2))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [22]:
rng = random.Random(2240)
for _ in range(500):
    total, c1, c2 = rng.randint(1, 120), rng.randint(1, 30), rng.randint(1, 30)
    exp = ways_brute(total, c1, c2)
    for name, fn in fns.items():
        assert fn(total, c1, c2) == exp, (name, total, c1, c2)
for _ in range(100):
    total, c1, c2 = rng.randint(1, 10**6), rng.randint(1, 10**6), rng.randint(1, 10**6)
    if total // max(c1, c2) > 2 * 10**4: continue
    vals = {name: fn(total, c1, c2) for name, fn in fns.items()}
    assert len(set(vals.values())) == 1, vals
for T in range(0, 50):
    assert ways_floor_sum(T, 1, 1) == (T + 1) * (T + 2) // 2
print("Ngẫu nhiên nhỏ + lớn + công thức đóng: OK")

Ngẫu nhiên nhỏ + lớn + công thức đóng: OK


### 12.1 Kiểm tra tính chất (property-based)

Ngoài so với brute force, kiểm các tính chất mà mọi lời giải đúng phải thỏa:

1. **Đối xứng:** `f(T, c1, c2) = f(T, c2, c1)`.
2. **Đơn điệu theo ngân sách:** `f(T + 1, c1, c2) ≥ f(T, c1, c2)` (thêm tiền không làm mất cách mua nào).
3. **Đơn điệu theo giá:** `f(T, c1 + 1, c2) ≤ f(T, c1, c2)`.
4. **Hiệu theo ngân sách** `f(T) − f(T − 1)` = số cách tiêu **đúng** `T` (bài tập 5) — liên hệ giữa "≤ T" và "= T".
5. **Giá vượt ngân sách:** nếu `c1 > T` và `c2 > T` thì đáp án là 1.

In [23]:
rngp = random.Random(99)
for _ in range(400):
    T, a, b = rngp.randint(1, 300), rngp.randint(1, 40), rngp.randint(1, 40)
    f = ways_floor_sum
    assert f(T, a, b) == f(T, b, a)
    assert f(T + 1, a, b) >= f(T, a, b)
    assert f(T, a + 1, b) <= f(T, a, b)
    assert f(T, a, b) - f(T - 1, a, b) == sum(1 for x in range(T // a + 1) if (T - a * x) % b == 0)
    assert f(T, T + 1 + a, T + 1 + b) == 1
print("5 tính chất đều thỏa trên 400 bộ ngẫu nhiên")

5 tính chất đều thỏa trên 400 bộ ngẫu nhiên


Tính chất 4 rất hữu ích: nó cho phép kiểm **chéo** lời giải "≤ T" với lời giải "= T" mà không cần brute force — một kỹ thuật kiểm thử mạnh khi brute force quá chậm.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên cộng 1 (bỏ trường hợp mua 0 bút chì)

In [24]:
def wrong_no_plus_one(total, cost1, cost2):
    return sum((total - cost1 * x) // cost2 for x in range(total // cost1 + 1))   # SAI: thiếu +1

print("ví dụ 1: sai =", wrong_no_plus_one(20, 10, 5), "| đúng = 9")
print("ví dụ 2: sai =", wrong_no_plus_one(5, 10, 10), "| đúng = 1")

ví dụ 1: sai = 6 | đúng = 9
ví dụ 2: sai = 0 | đúng = 1


In [25]:
print("sửa:", ways_one_loop(20, 10, 5), ways_one_loop(5, 10, 10))

sửa: 9 1


### Lỗi 2 — `range(total // cost1)` bỏ sót giá trị `x` lớn nhất

In [26]:
def wrong_range(total, cost1, cost2):
    return sum((total - cost1 * x) // cost2 + 1 for x in range(total // cost1))   # SAI: thiếu x = total//cost1

print("ví dụ 1: sai =", wrong_range(20, 10, 5), "| đúng = 9")
print("ví dụ 2: sai =", wrong_range(5, 10, 10), "| đúng = 1  (vòng lặp rỗng!)")

ví dụ 1: sai = 8 | đúng = 9
ví dụ 2: sai = 0 | đúng = 1  (vòng lặp rỗng!)


In [27]:
print("sửa (range(total // cost1 + 1)):", ways_one_loop(20, 10, 5), ways_one_loop(5, 10, 10))

sửa (range(total // cost1 + 1)): 9 1


### Lỗi 3 — Nhân hai số lượng tối đa (đếm hình chữ nhật thay vì tam giác)

`(T//c1 + 1) · (T//c2 + 1)` đếm cả các cặp vượt ngân sách (mua tối đa cả hai loại cùng lúc).

In [28]:
def wrong_rectangle(total, cost1, cost2):
    return (total // cost1 + 1) * (total // cost2 + 1)                             # SAI

print("ví dụ 1: sai =", wrong_rectangle(20, 10, 5), "| đúng = 9")

ví dụ 1: sai = 15 | đúng = 9


In [29]:
print("sửa:", ways_one_loop(20, 10, 5))

sửa: 9


### Lỗi 4 — Chia số thực rồi làm tròn

`int(T / c2)` dùng float (53 bit định trị): với ràng buộc gốc `T ≤ 10^6` thì vô hại, nhưng ở biến thể `T ~ 10^18` (khi dùng công thức `O(1)` hay floor_sum) phép chia float mất chính xác. Tìm phản ví dụ với `T` lớn cho công thức cấp số cộng của Cách 5.

In [30]:
def wrong_float(total, cost1, cost2):
    """Công thức cấp số cộng (Cách 5, khi cost2 | cost1) nhưng chia bằng float."""
    q = cost1 // cost2
    N = int(total / cost1)                                                       # SAI: float
    first = int(total / cost2) + 1                                               # SAI: float
    return (N + 1) * first - q * N * (N + 1) // 2

rngf = random.Random(4); found = None
for _ in range(5000):
    b = rngf.randint(1, 50); a = b * rngf.randint(1, 20); T = rngf.randint(10**17, 10**18)
    if wrong_float(T, a, b) != ways_divisible(T, a, b):
        found = (T, a, b); break
print("phản ví dụ:", found, "| T/c2 float =", int(found[0] / found[2]), "| T//c2 đúng =", found[0] // found[2])
print("sai =", wrong_float(*found), "| đúng =", ways_divisible(*found))

phản ví dụ: (931506476402546375, 160, 16) | T/c2 float = 58219154775159152 | T//c2 đúng = 58219154775159148
sai = 169473499136696869454819211563448 | đúng = 169473499136696846167157301499785


In [31]:
print("sửa (chỉ dùng //):", ways_divisible(*found), "| floor_sum:", ways_floor_sum(*found))

sửa (chỉ dùng //): 169473499136696846167157301499785 | floor_sum: 169473499136696846167157301499785


### Lỗi 5 — Dừng vòng lặp khi không mua được bút chì nào

Nghĩ "khi `rem < cost2` thì hết cách" và `break` — nhưng vẫn còn **1 cách** (mua 0 bút chì) cho mỗi `x` như vậy.

In [32]:
def wrong_break(total, cost1, cost2):
    ans = 0
    for x in range(total // cost1 + 1):
        rem = total - cost1 * x
        if rem < cost2:
            break                                                                # SAI: vẫn có 1 cách (y = 0)
        ans += rem // cost2 + 1
    return ans

bad = next((t, a, b) for t in range(1, 30) for a in range(1, 10) for b in range(1, 10) if wrong_break(t, a, b) != ways_brute(t, a, b))
print("phản ví dụ:", bad, "| sai =", wrong_break(*bad), "| đúng =", ways_brute(*bad))

phản ví dụ: (1, 1, 1) | sai = 2 | đúng = 3


In [33]:
print("sửa:", ways_one_loop(*bad))

sửa: 3


### 13.6 Lỗi 6 — Hoán đổi `cost1, cost2` nhưng vẫn dùng biến cũ ở công thức

Khi "duyệt theo món đắt hơn", bản sai hoán đổi để tính **số vòng** nhưng công thức chiều cao cột vẫn chia cho giá **cũ** — trộn lẫn hai cách đếm.

In [34]:
def wrong_swap_mix(total, cost1, cost2):
    big, small = max(cost1, cost2), min(cost1, cost2)
    return sum((total - cost1 * x) // cost2 + 1 for x in range(total // big + 1))   # SAI: dùng cost1/cost2 cũ

bad6 = next((t, a, b) for t in range(1, 30) for a in range(1, 8) for b in range(1, 8) if wrong_swap_mix(t, a, b) != ways_brute(t, a, b))
print("phản ví dụ:", bad6, "| sai =", wrong_swap_mix(*bad6), "| đúng =", ways_brute(*bad6))

phản ví dụ: (1, 1, 2) | sai = 1 | đúng = 2


In [35]:
print("sửa (hoán đổi rồi dùng nhất quán):", ways_pricier(*bad6))

sửa (hoán đổi rồi dùng nhất quán): 2


### 13.7 Lỗi 7 — floor_sum với `n` sai (lệch 1)

Gọi `floor_sum(N, …)` thay vì `floor_sum(N + 1, …)` bỏ cột `x = 0` (sau khi đổi biến `i = N − x`, đó là `i = N` — số hạng cuối). Lỗi lệch 1 kiểu này rất hay gặp khi đổi biến.

In [36]:
def wrong_floor_sum_n(total, cost1, cost2):
    N, r = divmod(total, cost1)
    return floor_sum(N, cost2, cost1, r) + (N + 1)                                # SAI: phải là N + 1 số hạng

print("ví dụ 1: sai =", wrong_floor_sum_n(20, 10, 5), "| đúng =", ways_floor_sum(20, 10, 5))

ví dụ 1: sai = 5 | đúng = 9


In [37]:
print("sửa:", ways_floor_sum(20, 10, 5))

sửa: 9


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cặp `(x, y) ≥ 0` thỏa `a·x + b·y ≤ T`" ⇒ cố định một biến, biến kia đếm bằng `//` + 1.
- Luôn kiểm trường hợp **mua 0** (đề nói "or none").
- Đối xứng ⇒ duyệt theo tham số lớn hơn.
- Ràng buộc `T ≤ 10^6` ⇒ `O(T)` đủ; nếu `T ~ 10^18` ⇒ floor_sum.
- Đáp án ~`5·10^11` ⇒ C++/Java cần `long long` / `long`.

## 15. Biến thể của bài

1. **Ba loại hàng** `a·x + b·y + c·z ≤ T`: cố định `x`, gọi hàm hai loại cho phần còn lại ⇒ `O(T/a · T/b)` hoặc `O(T/a · log)` với floor_sum.
2. **Tiêu đúng hết tiền** `a·x + b·y = T`: đếm nghiệm phương trình Diophantine — cố định `x`, kiểm `(T − a·x) % b == 0`; hoặc dùng `gcd` + Euclid mở rộng (bài tập 5).
3. **Giới hạn số lượng** `x ≤ X, y ≤ Y`: lấy `min` trong công thức chiều cao cột.
4. **LeetCode 518 — Coin Change II:** nhiều loại, tiêu đúng `T` ⇒ quy hoạch động.
5. **Đếm điểm nguyên dưới đường thẳng** trong hình học tính toán — floor_sum / định lý Pick.

Minh họa biến thể 1 (tự dựng, kiểm bằng brute force):

In [38]:
def ways_three(total, a, b, c):
    return sum(ways_floor_sum(total - a * x, b, c) for x in range(total // a + 1))

def ways_three_brute(total, a, b, c):
    return sum(1 for x in range(total // a + 1) for y in range(total // b + 1) for z in range(total // c + 1) if a * x + b * y + c * z <= total)

rng3 = random.Random(3)
for _ in range(200):
    t, a, b, c = rng3.randint(1, 40), rng3.randint(1, 9), rng3.randint(1, 9), rng3.randint(1, 9)
    assert ways_three(t, a, b, c) == ways_three_brute(t, a, b, c)
print("biến thể 1 khớp brute; ways_three(20, 10, 5, 4) =", ways_three(20, 10, 5, 4))

biến thể 1 khớp brute; ways_three(20, 10, 5, 4) = 23


Minh họa biến thể 4 (Coin Change II — nhiều loại tiền, tiêu **đúng** `T`): với đúng hai loại giá, quy hoạch động phải khớp với `ways_exact` ở bài tập 5 và với hiệu `f(T) − f(T−1)` của bài gốc.

In [39]:
def coin_change_ways(T, coins):
    dp = [1] + [0] * T
    for c in coins:
        for s in range(c, T + 1):
            dp[s] += dp[s - c]
    return dp[T]

rngc = random.Random(12)
for _ in range(200):
    T, a, b = rngc.randint(1, 150), rngc.randint(1, 20), rngc.randint(1, 20)
    if a == b: continue
    assert coin_change_ways(T, [a, b]) == ways_floor_sum(T, a, b) - ways_floor_sum(T - 1, a, b)
print("Coin Change II với 2 loại = f(T) - f(T-1); ví dụ T=20, [10, 5]:", coin_change_ways(20, [10, 5]))

Coin Change II với 2 loại = f(T) - f(T-1); ví dụ T=20, [10, 5]: 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng | thử mọi cặp | `O(T²/(c1·c2))` | `O(1)` | kiểm thử |
| 2. Một vòng | cột cao `⌊rem/c2⌋ + 1` | `O(T/c1)` | `O(1)` | dễ viết |
| 3. Món đắt hơn | như 2, đổi vai | `O(T/max)` | `O(1)` | **nộp bài** |
| 4. floor_sum | Euclid trên tổng sàn | `O(log)` | `O(1)` | `T` khổng lồ |
| 5. Cấp số cộng | khi `c2 | c1` | `O(1)` | `O(1)` | trường hợp đặc biệt |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `waysToBuyPensPencils(10, 3, 4)`.
2. **Tính tay:** bảng cột cho `total = 7, cost1 = 2, cost2 = 3`.
3. **Điền TODO:** `ways_capped(total, c1, c2, X, Y)` — mua tối đa `X` bút mực, `Y` bút chì.
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** `ways_exact(total, c1, c2)` — số cách **tiêu đúng hết** `total`. Kiểm bằng brute force.

In [40]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def ways_capped(total, c1, c2, X, Y):
    # TODO
    pass

In [41]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_ways(total, cost1, cost2):
    # TODO
    pass

In [42]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def ways_exact(total, c1, c2):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `x = 0..3`. (2) Cột: `10//4+1 = 3`, `7//4+1 = 2`, `4//4+1 = 2`, `1//4+1 = 1`. (3) Tổng `8`.

**Bài 2.** (1) `x = 0..3`. (2) Tiền còn `7, 5, 3, 1` ⇒ cột `3, 2, 2, 1`. (3) Tổng `8`.

**Bài 3.** (1) `x` chạy tới `min(X, total // c1)`. (2) Cột cao `min(Y, rem // c2) + 1`. (3) Kiểm bằng brute.

**Bài 4.** (1) Vòng `x`. (2) `rem // cost2 + 1`. (3) `range(total // cost1 + 1)`.

**Bài 5.** (1) Cố định `x`, cần `(total − c1·x) % c2 == 0`. (2) Nhanh hơn: các `x` thỏa mãn tạo cấp số cộng với công sai `c2 / gcd(c1, c2)`. (3) Tìm `x` nhỏ nhất bằng duyệt `0..c2/g − 1` hoặc nghịch đảo modulo.

## 19. Đáp án tham khảo

In [43]:
# Bài 1, 2
assert Solution().waysToBuyPensPencils(10, 3, 4) == 8 == ways_brute(10, 3, 4)
assert trace_one_loop(7, 2, 3) == 8 == ways_brute(7, 2, 3)
print("Bài 1, 2: OK")

x=0: còn 7, bút chì 0..2 -> +3, ans = 3
x=1: còn 5, bút chì 0..1 -> +2, ans = 5
x=2: còn 3, bút chì 0..1 -> +2, ans = 7
x=3: còn 1, bút chì 0..0 -> +1, ans = 8
Bài 1, 2: OK


In [44]:
# Bài 3, 4
def ways_capped_ans(total, c1, c2, X, Y):
    return sum(min(Y, (total - c1 * x) // c2) + 1 for x in range(min(X, total // c1) + 1))

def ways_capped_brute(total, c1, c2, X, Y):
    return sum(1 for x in range(X + 1) for y in range(Y + 1) if c1 * x + c2 * y <= total)

my_ways_ans = ways_one_loop
rng4 = random.Random(44)
for _ in range(300):
    t, a, b, X, Y = rng4.randint(1, 60), rng4.randint(1, 12), rng4.randint(1, 12), rng4.randint(0, 8), rng4.randint(0, 8)
    assert ways_capped_ans(t, a, b, X, Y) == ways_capped_brute(t, a, b, X, Y)
    assert my_ways_ans(t, a, b) == ways_brute(t, a, b)
print("Bài 3, 4: OK; ways_capped(20, 10, 5, 1, 1) =", ways_capped_ans(20, 10, 5, 1, 1))

Bài 3, 4: OK; ways_capped(20, 10, 5, 1, 1) = 4


In [45]:
# Bài 5
from math import gcd

def ways_exact_ans(total, c1, c2):
    g = gcd(c1, c2)
    if total % g: return 0
    step = c2 // g                                   # các x hợp lệ cách nhau step
    x0 = next((x for x in range(step) if (total - c1 * x) % c2 == 0), None)
    if x0 is None or c1 * x0 > total: return 0
    return (total // c1 - x0) // step + 1

def ways_exact_brute(total, c1, c2):
    return sum(1 for x in range(total // c1 + 1) if (total - c1 * x) % c2 == 0)

rng5 = random.Random(55)
for _ in range(500):
    t, a, b = rng5.randint(1, 200), rng5.randint(1, 20), rng5.randint(1, 20)
    assert ways_exact_ans(t, a, b) == ways_exact_brute(t, a, b), (t, a, b)
print("Bài 5: OK; ways_exact(20, 10, 5) =", ways_exact_ans(20, 10, 5), "(x=0,y=4; x=1,y=2; x=2,y=0)")

Bài 5: OK; ways_exact(20, 10, 5) = 3 (x=0,y=4; x=1,y=2; x=2,y=0)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
if c1 < c2: c1, c2 = c2, c1
ans = 0
for x in range(total // c1 + 1):
    ans += (total - c1 * x) // c2 + 1
```

In [46]:
def template(total, c1, c2):
    if c1 < c2: c1, c2 = c2, c1
    return sum((total - c1 * x) // c2 + 1 for x in range(total // c1 + 1))
print([template(t, a, b) for t, a, b, _ in examples])

[9, 1]


**Checklist:**

- [ ] `range(total // c1 + 1)` — có cả `x` lớn nhất.
- [ ] Cộng `+1` cho trường hợp mua 0 món thứ hai.
- [ ] Không `break` khi `rem < c2` (vẫn còn 1 cách).
- [ ] Chỉ dùng `//`, không chia float.
- [ ] Đếm tam giác, không phải hình chữ nhật.
- [ ] Ngôn ngữ khác: kiểu 64-bit cho đáp án.